# Atividade prática em OLAP

- **Alunos:** Maruan Biasi El Achkar e Ricardo Falcão Schlieper
- **Professor:** Luiz Carlos Camargo
- **Matéria**: Análise Preditiva
- **Repositório**: https://github.com/CodyKoInABox/atividade-olap-camargo

## 1. Instalar e iniciar MySQL

In [12]:
!apt-get -y update > /dev/null
!DEBIAN_FRONTEND=noninteractive apt-get install -y mysql-server > /dev/null

!service mysql start
!mysql --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
 * Starting MySQL database server mysqld
   ...done.
mysql  Ver 8.0.45-0ubuntu0.22.04.1 for Linux on x86_64 ((Ubuntu))


## 2. Criar banco e tabelas

De acordo com o PDF da atividade:  
- `Estudante(estudanteID, nome, curso)`  
- `Instrutor(instrutorID, curso)`  
- `Aula(aulaID, instituicao, cidade, estado)`  
- `Aulas_assistidas(estudanteID, instrutorID, aulaID, notas)`

In [13]:
sql = """
DROP DATABASE IF EXISTS olap_atividade;
CREATE DATABASE olap_atividade;
USE olap_atividade;

CREATE TABLE Estudante (
    estudanteID INT PRIMARY KEY,
    nome VARCHAR(100) NOT NULL,
    curso VARCHAR(100) NOT NULL
);

CREATE TABLE Instrutor (
    instrutorID INT PRIMARY KEY,
    curso VARCHAR(100) NOT NULL
);

CREATE TABLE Aula (
    aulaID INT PRIMARY KEY,
    instituicao VARCHAR(100) NOT NULL,
    cidade VARCHAR(100) NOT NULL,
    estado CHAR(2) NOT NULL
);

CREATE TABLE Aulas_assistidas (
    estudanteID INT NOT NULL,
    instrutorID INT NOT NULL,
    aulaID INT NOT NULL,
    notas DECIMAL(5,2) NOT NULL,
    PRIMARY KEY (estudanteID, instrutorID, aulaID),
    FOREIGN KEY (estudanteID) REFERENCES Estudante(estudanteID),
    FOREIGN KEY (instrutorID) REFERENCES Instrutor(instrutorID),
    FOREIGN KEY (aulaID) REFERENCES Aula(aulaID)
);
"""
with open('/tmp/setup.sql', 'w') as f:
    f.write(sql)
!mysql -u root < /tmp/setup.sql

## 3. Inserir dados de exemplo


In [14]:
sql = r'''
USE olap_atividade;

INSERT INTO Estudante (estudanteID, nome, curso) VALUES
(1, 'Icaro', 'Computacao'),
(2, 'Luquinhas', 'Psicologia'),
(3, 'Gustavo', 'Medicina'),
(4, 'Habib', 'Linguas'),
(5, 'Murilo', 'Psicologia');

INSERT INTO Instrutor (instrutorID, curso) VALUES
(10, 'Computacao'),
(11, 'Medicina'),
(12, 'Psicologia'),
(13, 'Linguas');

INSERT INTO Aula (aulaID, instituicao, cidade, estado) VALUES
(100, 'UDESC', 'Joinville', 'SC'),
(101, 'UFSC', 'Florianopolis', 'SC'),
(102, 'PUCPR', 'Curitiba', 'PR'),
(103, 'Univille', 'Joinville', 'SC'),
(104, 'USP', 'Sao Paulo', 'SP');

INSERT INTO Aulas_assistidas (estudanteID, instrutorID, aulaID, notas) VALUES
(1, 11, 100, 82.0),
(1, 10, 103, 91.0),
(2, 10, 100, 77.0),
(2, 11, 102, 88.0),
(3, 12, 101, 93.0),
(3, 10, 103, 69.0),
(4, 11, 103, 72.0),
(4, 10, 100, 64.0),
(5, 13, 104, 95.0),
(5, 11, 101, 71.0),
(2, 13, 103, 67.0),
(1, 12, 101, 74.0);
'''
open('/tmp/data.sql', 'w').write(sql)
!mysql -u root < /tmp/data.sql
print("Dados inseridos")

Dados inseridos


## 4. Visualizar dados

In [15]:
!mysql -u root -D olap_atividade -e "SELECT * FROM Estudante;"
!mysql -u root -D olap_atividade -e "SELECT * FROM Instrutor;"
!mysql -u root -D olap_atividade -e "SELECT * FROM Aula;"
!mysql -u root -D olap_atividade -e "SELECT * FROM Aulas_assistidas;"

+-------------+-----------+------------+
| estudanteID | nome      | curso      |
+-------------+-----------+------------+
|           1 | Icaro     | Computacao |
|           2 | Luquinhas | Psicologia |
|           3 | Gustavo   | Medicina   |
|           4 | Habib     | Linguas    |
|           5 | Murilo    | Psicologia |
+-------------+-----------+------------+
+-------------+------------+
| instrutorID | curso      |
+-------------+------------+
|          10 | Computacao |
|          11 | Medicina   |
|          12 | Psicologia |
|          13 | Linguas    |
+-------------+------------+
+--------+-------------+---------------+--------+
| aulaID | instituicao | cidade        | estado |
+--------+-------------+---------------+--------+
|    100 | UDESC       | Joinville     | SC     |
|    101 | UFSC        | Florianopolis | SC     |
|    102 | PUCPR       | Curitiba      | PR     |
|    103 | Univille    | Joinville     | SC     |
|    104 | USP         | Sao Paulo     | SP     |

## 5. Query a

**Encontre todos os alunos que tiveram aulas em Santa Catarina com um instrutor que não pertence ao curso do aluno e obtiveram pontuação acima de 65. Retorne nome do aluno, universidade e pontuação.**

In [16]:
query_a = """
SELECT e.nome AS aluno, a.instituicao AS universidade, aa.notas AS pontuacao
FROM Aulas_assistidas aa
JOIN Estudante e ON aa.estudanteID = e.estudanteID
JOIN Instrutor i ON aa.instrutorID = i.instrutorID
JOIN Aula a ON aa.aulaID = a.aulaID
WHERE a.estado = 'SC' AND i.curso <> e.curso AND aa.notas > 65
ORDER BY e.nome, a.instituicao;
"""
with open('/tmp/query_a.sql', 'w') as f:
    f.write("USE olap_atividade;\n" + query_a)
!mysql -u root < /tmp/query_a.sql

aluno	universidade	pontuacao
Gustavo	UFSC	93.00
Gustavo	Univille	69.00
Habib	Univille	72.00
Icaro	UDESC	82.00
Icaro	UFSC	74.00
Luquinhas	UDESC	77.00
Luquinhas	Univille	67.00
Murilo	UFSC	71.00


## 6. Query b

**Encontre as pontuações médias agrupadas por aluno e por instrutor dos cursos ministrados em Joinville.**

In [17]:
query_b = '''
SELECT e.nome AS aluno,
       aa.instrutorID,
       AVG(aa.notas) AS media_notas
FROM Aulas_assistidas aa
JOIN Estudante e ON aa.estudanteID = e.estudanteID
JOIN Aula a ON aa.aulaID = a.aulaID
WHERE a.cidade = 'Joinville'
GROUP BY e.nome, aa.instrutorID
ORDER BY e.nome, aa.instrutorID;
'''
open('/tmp/query_b.sql', 'w').write("USE olap_atividade;\n" + query_b)
!mysql -u root < /tmp/query_b.sql

aluno	instrutorID	media_notas
Gustavo	10	69.000000
Habib	10	64.000000
Habib	11	72.000000
Icaro	10	91.000000
Icaro	11	82.000000
Luquinhas	10	77.000000
Luquinhas	13	67.000000


## 7. Query c

**A partir do resultado da query b, fazer rollup para agrupar somente por instrutor.**  

In [18]:
query_c = """
SELECT
    CASE WHEN aa.instrutorID IS NULL THEN 'TOTAL' ELSE CAST(aa.instrutorID AS CHAR) END AS instrutor,
    AVG(aa.notas) AS media
FROM Aulas_assistidas aa
JOIN Aula a ON aa.aulaID = a.aulaID
WHERE a.cidade = 'Joinville'
GROUP BY aa.instrutorID WITH ROLLUP;
"""
with open('/tmp/query_c.sql', 'w') as f:
    f.write("USE olap_atividade;\n" + query_c)
!mysql -u root < /tmp/query_c.sql

instrutor	media
10	75.250000
11	77.000000
13	67.000000
TOTAL	74.571429


## 8. Query d

**Encontrar a média de pontuação por curso do estudante.**

In [19]:
query_d = '''
SELECT e.curso AS curso_estudante,
       AVG(aa.notas) AS media_notas
FROM Aulas_assistidas aa
JOIN Estudante e ON aa.estudanteID = e.estudanteID
GROUP BY e.curso
ORDER BY e.curso;
'''
open('/tmp/query_d.sql', 'w').write("USE olap_atividade;\n" + query_d)
!mysql -u root < /tmp/query_d.sql

curso_estudante	media_notas
Computacao	82.333333
Linguas	68.000000
Medicina	81.000000
Psicologia	79.600000


## 9. Query e

**Drill down com base na query d: agrupar pelo curso do instrutor e também pelo curso do estudante. **

In [20]:
query_e = '''
SELECT e.curso AS curso_estudante,
       i.curso AS curso_instrutor,
       AVG(aa.notas) AS media_notas
FROM Aulas_assistidas aa
JOIN Estudante e ON aa.estudanteID = e.estudanteID
JOIN Instrutor i ON aa.instrutorID = i.instrutorID
GROUP BY e.curso, i.curso
ORDER BY e.curso, i.curso;
'''
open('/tmp/query_e.sql', 'w').write("USE olap_atividade;\n" + query_e)
!mysql -u root < /tmp/query_e.sql

curso_estudante	curso_instrutor	media_notas
Computacao	Computacao	91.000000
Computacao	Medicina	82.000000
Computacao	Psicologia	74.000000
Linguas	Computacao	64.000000
Linguas	Medicina	72.000000
Medicina	Computacao	69.000000
Medicina	Psicologia	93.000000
Psicologia	Computacao	77.000000
Psicologia	Linguas	81.000000
Psicologia	Medicina	79.500000


## 10. Query f

**Usar `WITH ROLLUP` nos atributos da tabela Aula para obter pontuações médias nas granularidades geográficas: estado, cidade e instituição, além da média geral.**

In [21]:
query_f = """
SELECT
    COALESCE(a.estado, 'T_ESTADO') AS estado,
    COALESCE(a.cidade, 'T_CIDADE') AS cidade,
    AVG(aa.notas) AS media
FROM Aulas_assistidas aa
JOIN Aula a ON aa.aulaID = a.aulaID
GROUP BY a.estado, a.cidade WITH ROLLUP;
"""
with open('/tmp/query_f.sql', 'w') as f:
    f.write("USE olap_atividade;\n" + query_f)
!mysql -u root < /tmp/query_f.sql

estado	cidade	media
PR	Curitiba	88.000000
PR	T_CIDADE	88.000000
SC	Florianopolis	79.333333
SC	Joinville	74.571429
SC	T_CIDADE	76.000000
SP	Sao Paulo	95.000000
SP	T_CIDADE	95.000000
T_ESTADO	T_CIDADE	78.583333


## 11. Query g

**Criar um cubo; verifique se o SGBD em uso possui o operador cube, se afirmativo, utilize, caso contrário, tente criar com os operadores union e rollup.**

In [22]:
query_g = '''
SELECT
    e.curso AS curso_estudante,
    i.curso AS curso_instrutor,
    AVG(aa.notas) AS media_notas,
    'DETALHE' AS nivel
FROM Aulas_assistidas aa
JOIN Estudante e ON aa.estudanteID = e.estudanteID
JOIN Instrutor i ON aa.instrutorID = i.instrutorID
GROUP BY e.curso, i.curso

UNION ALL

SELECT
    e.curso AS curso_estudante,
    'TODOS' AS curso_instrutor,
    AVG(aa.notas) AS media_notas,
    'SUBTOTAL_ESTUDANTE' AS nivel
FROM Aulas_assistidas aa
JOIN Estudante e ON aa.estudanteID = e.estudanteID
GROUP BY e.curso

UNION ALL

SELECT
    'TODOS' AS curso_estudante,
    i.curso AS curso_instrutor,
    AVG(aa.notas) AS media_notas,
    'SUBTOTAL_INSTRUTOR' AS nivel
FROM Aulas_assistidas aa
JOIN Instrutor i ON aa.instrutorID = i.instrutorID
GROUP BY i.curso

UNION ALL

SELECT
    'TODOS' AS curso_estudante,
    'TODOS' AS curso_instrutor,
    AVG(notas) AS media_notas,
    'TOTAL_GERAL' AS nivel
FROM Aulas_assistidas
ORDER BY nivel, curso_estudante, curso_instrutor;
'''
open('/tmp/query_g.sql', 'w').write("USE olap_atividade;\n" + query_g)
!mysql -u root < /tmp/query_g.sql

curso_estudante	curso_instrutor	media_notas	nivel
Computacao	Computacao	91.000000	DETALHE
Computacao	Medicina	82.000000	DETALHE
Computacao	Psicologia	74.000000	DETALHE
Linguas	Computacao	64.000000	DETALHE
Linguas	Medicina	72.000000	DETALHE
Medicina	Computacao	69.000000	DETALHE
Medicina	Psicologia	93.000000	DETALHE
Psicologia	Computacao	77.000000	DETALHE
Psicologia	Linguas	81.000000	DETALHE
Psicologia	Medicina	79.500000	DETALHE
Computacao	TODOS	82.333333	SUBTOTAL_ESTUDANTE
Linguas	TODOS	68.000000	SUBTOTAL_ESTUDANTE
Medicina	TODOS	81.000000	SUBTOTAL_ESTUDANTE
Psicologia	TODOS	79.600000	SUBTOTAL_ESTUDANTE
TODOS	Computacao	75.250000	SUBTOTAL_INSTRUTOR
TODOS	Linguas	81.000000	SUBTOTAL_INSTRUTOR
TODOS	Medicina	78.250000	SUBTOTAL_INSTRUTOR
TODOS	Psicologia	83.500000	SUBTOTAL_INSTRUTOR
TODOS	TODOS	78.583333	TOTAL_GERAL
